# Signal Fundamentals: Cosine, Noise, FFT, and Spectrogram

**Topics covered:**
- Generating and visualising real discrete-time sinusoidal signals
- Complex baseband (IQ) representation — the foundation of all wireless systems
- Modelling Additive White Gaussian Noise (AWGN) and controlling SNR
- Computing the FFT and interpreting the two-sided power spectrum
- Building and interpreting a spectrogram (STFT) and the time-frequency resolution trade-off

**Author:** Sohail Payami

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import chirp, spectrogram

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.4,
    'lines.linewidth': 1.2,
})

## 1. Real Discrete-Time Cosine Signal

The simplest test signal in DSP. In MATLAB you would write `t = 0:1/Fs:T-1/Fs;` and `x = cos(2*pi*f0*t);`.
NumPy's `arange` and `cos` are direct equivalents.

$$x[n] = A \cos\!\left(2\pi f_0 \frac{n}{F_s} + \phi\right), \quad n = 0, 1, \ldots, N-1$$

where $F_s$ is the sampling frequency (Hz) and $f_0$ is the tone frequency.
The **Nyquist criterion** requires $f_0 < F_s / 2$ — any frequency above half the sampling rate aliases back into the baseband.

In [ ]:
Fs = 10_000          # sampling frequency (Hz) — MATLAB: Fs = 10000
f0 = 150             # tone frequency (Hz), well below the Nyquist limit of 5 kHz
duration = 0.1       # signal duration (s) → N = Fs * duration samples

t = np.arange(0, duration, 1/Fs)    # time vector — MATLAB: t = 0:1/Fs:duration-1/Fs
x = np.cos(2 * np.pi * f0 * t)     # real cosine
N = len(t)

print(f'Samples: {N}  |  Fs: {Fs} Hz  |  Nyquist limit: {Fs//2} Hz  |  f0: {f0} Hz')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t * 1e3, x)
ax.set(xlabel='Time (ms)', ylabel='Amplitude',
       title=f'Real Cosine — f0 = {f0} Hz, Fs = {Fs} Hz')
plt.tight_layout()
plt.show()

## 2. Complex Baseband — The IQ Representation

In all modern wireless systems (LTE, 5G NR, Wi-Fi), signals are processed at **complex baseband** rather than as real bandpass waveforms.
The receiver down-converts the RF signal to produce two streams:

- **I (In-phase):** the real part
- **Q (Quadrature):** the imaginary part, 90° phase-shifted relative to I

A real bandpass cosine at carrier frequency $f_c$ has a complex baseband equivalent — a rotating **phasor**:

$$\tilde{x}[n] = e^{j 2\pi f_c n / F_s} = \underbrace{\cos(2\pi f_c n/F_s)}_{I} + j\underbrace{\sin(2\pi f_c n/F_s)}_{Q}$$

The imaginary unit in Python/NumPy is `1j` — **same syntax as MATLAB**.
In the IQ plane, this signal traces a perfect unit circle, rotating at $f_c$ revolutions per second.
The **envelope** $|\tilde{x}[n]| = 1$ is constant — there is no amplitude modulation here.

In [ ]:
x_iq = np.exp(1j * 2 * np.pi * f0 * t)   # complex phasor — MATLAB: exp(1j*2*pi*f0*t)

fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
axes[0].plot(t * 1e3, x_iq.real, color='steelblue',  label='I  (real part)')
axes[1].plot(t * 1e3, x_iq.imag, color='darkorange', label='Q  (imaginary part)')
axes[2].plot(t * 1e3, np.abs(x_iq), color='green',   label='|I + jQ|  (envelope = 1)')
for ax in axes:
    ax.legend(loc='upper right')
    ax.set_ylabel('Amplitude')
axes[2].set_xlabel('Time (ms)')
axes[0].set_title('Complex Baseband Signal — I, Q, and Envelope')
plt.tight_layout()
plt.show()

# Phasor trajectory in the IQ plane — a perfect unit circle for constant-amplitude signals
fig, ax = plt.subplots(figsize=(4, 4), subplot_kw={'aspect': 'equal'})
theta = np.linspace(0, 2*np.pi, 300)
ax.plot(np.cos(theta), np.sin(theta), 'k--', alpha=0.3, linewidth=0.8)
ax.scatter(x_iq.real[::5], x_iq.imag[::5], s=8, color='steelblue', alpha=0.7)
ax.set(xlabel='I', ylabel='Q', title='Phasor in IQ Plane (every 5th sample)')
plt.tight_layout()
plt.show()

## 3. Additive White Gaussian Noise (AWGN)

AWGN is the standard model for thermal receiver noise. The received signal is:

$$r[n] = x[n] + w[n], \qquad w[n] \sim \mathcal{N}(0,\, \sigma^2)$$

Noise samples are **i.i.d.** (independent, identically distributed) Gaussian with zero mean and variance $\sigma^2$.
"White" means the power spectral density is flat across all frequencies — every frequency bin sees the same noise power.

**Signal-to-Noise Ratio:**

$$\text{SNR}_{\text{dB}} = 10\log_{10}\!\left(\frac{P_{\text{signal}}}{\sigma^2}\right)$$

To generate noise at a **target SNR**, we measure the signal power, derive the required $\sigma^2$, and scale `randn` output accordingly:

- MATLAB: `noise = sqrt(sigma2) * randn(1, N);`
- Python: `noise = np.sqrt(sigma2) * np.random.randn(N)`

At SNR = 0 dB the noise power equals the signal power. Below 0 dB the noise dominates.

In [ ]:
def add_awgn(signal, snr_db):
    """Add AWGN to a signal at a specified SNR (dB). Returns (noisy_signal, noise)."""
    signal_power = np.mean(np.abs(signal) ** 2)
    noise_power  = signal_power / (10 ** (snr_db / 10))
    noise = np.sqrt(noise_power) * np.random.randn(len(signal))
    return signal + noise, noise

np.random.seed(42)

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
for ax, snr in zip(axes, [20, 10, 0]):
    x_noisy, _ = add_awgn(x, snr)
    ax.plot(t * 1e3, x_noisy, linewidth=0.7)
    ax.set_ylabel('Amplitude')
    ax.set_title(f'SNR = {snr} dB')
axes[-1].set_xlabel('Time (ms)')
plt.suptitle('Cosine + AWGN at Different SNR Levels', y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Numerically verify the noise scaling — measured SNR should match the target
header = '  Target SNR  |  Measured SNR'
print(header)
print('-' * len(header))
for snr_target in [20, 10, 0, -5]:
    _, noise = add_awgn(x, snr_target)
    measured = 10 * np.log10(np.mean(x**2) / np.mean(noise**2))
    print(f'  {snr_target:>8} dB  |  {measured:>9.2f} dB')

## 4. Frequency Domain — FFT and Power Spectrum

The Fast Fourier Transform (FFT) decomposes the signal into its sinusoidal frequency components.
NumPy's `fft` is identical in behaviour to MATLAB's `fft`.

**Key steps — same as MATLAB:**
1. `np.fft.fft(x)` — compute the DFT (output is complex, length $N$)
2. `np.fft.fftshift(...)` — shift the zero-frequency bin to the centre (≡ MATLAB `fftshift`)
3. `np.fft.fftfreq(N, 1/Fs)` — construct the frequency axis in Hz

A real cosine at $f_0$ appears as **two impulses at $\pm f_0$** in the two-sided spectrum — one for each complex exponential in Euler's formula.
When AWGN is added, a **flat noise floor** appears across all bins; its height is determined by $\sigma^2 / N$.

**Amplitude in dBFS** (dB relative to full scale): $\; 20\log_{10}|X[k]/N|$

Normalising by $N$ ensures that the peak amplitude of a unit-amplitude cosine reads 0 dBFS, matching the `fft(x)/N` MATLAB convention.

In [ ]:
def compute_spectrum(sig, Fs):
    """Two-sided amplitude spectrum in dBFS, normalised by N."""
    N     = len(sig)
    X     = np.fft.fftshift(np.fft.fft(sig)) / N       # MATLAB: fftshift(fft(x)) / N
    freqs = np.fft.fftshift(np.fft.fftfreq(N, 1/Fs))   # frequency axis in Hz
    mag   = 20 * np.log10(np.abs(X) + 1e-12)           # +1e-12 guards against log(0)
    return freqs, mag

np.random.seed(0)
x_noisy_10dB, _ = add_awgn(x, 10)

freqs, mag_clean = compute_spectrum(x, Fs)
_, mag_noisy     = compute_spectrum(x_noisy_10dB, Fs)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(freqs, mag_clean)
axes[0].set(title='Amplitude Spectrum — Clean Signal', ylabel='Magnitude (dBFS)')

axes[1].plot(freqs, mag_noisy, linewidth=0.8)
axes[1].set(title='Amplitude Spectrum — Noisy Signal (SNR = 10 dB)',
             ylabel='Magnitude (dBFS)', xlabel='Frequency (Hz)')

for ax in axes:
    ax.set_xlim([-500, 500])
    ax.axvline( f0, color='red', linestyle='--', alpha=0.6, linewidth=1, label=f'+{f0} Hz')
    ax.axvline(-f0, color='red', linestyle='--', alpha=0.6, linewidth=1, label=f'-{f0} Hz')
    ax.legend()
plt.tight_layout()
plt.show()

## 5. Spectrogram — Time-Frequency Analysis

### Why the FFT alone is not enough

The FFT gives a **single, time-averaged snapshot** of frequency content — it cannot reveal *when* a frequency appeared or how the spectrum evolves over time.
Many real-world signals have time-varying spectral structure:

- A radar chirp sweeping from low to high frequency
- A 5G NR slot where DMRS pilots occupy specific OFDM symbols
- A frequency-hopped (FHSS) signal

The FFT collapses all of this time structure into one averaged picture, making the sweep invisible.

### Short-Time Fourier Transform (STFT)

The STFT solves this by sliding a short analysis window $w[\tau]$ along the signal and computing the FFT at each position $m$:

$$X[m, k] = \sum_{\tau=0}^{L-1} x[mH + \tau]\, w[\tau]\, e^{-j2\pi k\tau / N_{\text{FFT}}}$$

where $L$ is the **window length**, $H$ is the **hop size** (stride between windows), and $k$ is the frequency bin index.
The result is a 2-D array indexed by time frame $m$ and frequency bin $k$.

The **spectrogram** is the squared magnitude:

$$S[m, k] = |X[m, k]|^2$$

### Time-frequency trade-off

There is a fundamental uncertainty principle (analogous to Heisenberg in quantum mechanics) that limits simultaneous time and frequency resolution:

$$\Delta t \cdot \Delta f \geq \frac{1}{4\pi}$$

| Window length | Time resolution | Frequency resolution |
|:---:|:---:|:---:|
| Short (e.g. 64 samples) | Fine — captures rapid changes | Poor — wide frequency bins |
| Long (e.g. 1024 samples) | Poor — temporal blurring | Fine — narrow frequency bins |

This is the same trade-off behind the 5G NR **subcarrier spacing (SCS)** choice:
15 kHz SCS uses a long OFDM symbol (~66.7 µs, fine frequency resolution), while 120 kHz SCS uses a short symbol (~8.3 µs, better time resolution for high-mobility channels).

### Chirp signal for demonstration

A static tone produces a flat horizontal line in the spectrogram — it teaches nothing about time-frequency structure.
Instead we use a **linear chirp** whose instantaneous frequency increases linearly with time:

$$f_i(t) = f_{\text{start}} + \frac{f_{\text{end}} - f_{\text{start}}}{T}\, t$$

This makes the time-varying structure immediately visible, and the trade-off between time and frequency resolution easy to observe.

In [ ]:
Fs_ch   = 8_000
T_ch    = 1.0
t_ch    = np.linspace(0, T_ch, int(Fs_ch * T_ch), endpoint=False)

f_start, f_end = 50, 2000
x_chirp = chirp(t_ch, f0=f_start, f1=f_end, t1=T_ch, method='linear')

np.random.seed(7)
x_chirp_noisy, _ = add_awgn(x_chirp, 20)   # light noise — SNR = 20 dB

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t_ch, x_chirp_noisy, linewidth=0.6)
ax.set(xlabel='Time (s)', ylabel='Amplitude',
       title=f'Linear Chirp: {f_start} Hz to {f_end} Hz over {T_ch} s  (SNR = 20 dB)')
plt.tight_layout()
plt.show()

In [ ]:
# Side-by-side comparison: FFT (time-averaged) vs spectrogram (time-resolved)
nperseg = 256          # window length — 256/8000 = 32 ms per frame
noverlap = nperseg // 2

f_sg, t_sg, Sxx = spectrogram(x_chirp_noisy, fs=Fs_ch,
                               nperseg=nperseg, noverlap=noverlap, window='hann')

freqs_ch, mag_ch = compute_spectrum(x_chirp_noisy, Fs_ch)

fig, axes = plt.subplots(3, 1, figsize=(12, 10))

axes[0].plot(t_ch, x_chirp_noisy, linewidth=0.5)
axes[0].set(xlabel='Time (s)', ylabel='Amplitude', title='Time Domain — Chirp Signal')

# The FFT averages over all time — the sweep is completely invisible in the spectrum
pos_mask = freqs_ch >= 0
axes[1].plot(freqs_ch[pos_mask], mag_ch[pos_mask])
axes[1].set(xlabel='Frequency (Hz)', ylabel='Magnitude (dBFS)',
             title='FFT — frequency content collapsed over all time (sweep structure is lost)',
             xlim=[0, 2500])

im = axes[2].pcolormesh(t_sg, f_sg, 10 * np.log10(Sxx + 1e-12),
                         shading='gouraud', cmap='viridis')
axes[2].set(xlabel='Time (s)', ylabel='Frequency (Hz)',
             title=f'Spectrogram (STFT) — window = {nperseg} samples ({nperseg/Fs_ch*1e3:.0f} ms), 50% overlap',
             ylim=[0, 2500])
fig.colorbar(im, ax=axes[2], label='Power (dB)')
plt.tight_layout()
plt.show()

In [ ]:
# Visualise the time-frequency trade-off by varying the analysis window length
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

for ax, nper in zip(axes, [64, 256, 1024]):
    f_s, t_s, S = spectrogram(x_chirp, fs=Fs_ch, nperseg=nper,
                               noverlap=nper // 2, window='hann')
    ax.pcolormesh(t_s, f_s, 10 * np.log10(S + 1e-12), shading='gouraud', cmap='viridis')
    freq_res    = Fs_ch / nper
    time_res_ms = nper / Fs_ch * 1e3
    ax.set(title=f'Window = {nper} samples ({time_res_ms:.1f} ms) | df={freq_res:.0f} Hz  dt={time_res_ms:.1f} ms',
           xlabel='Time (s)', ylim=[0, 2500])

axes[0].set_ylabel('Frequency (Hz)')
plt.suptitle('Time-Frequency Trade-off: Short window gives fine time resolution; Long window gives fine frequency resolution',
             fontsize=10, y=1.02)
plt.tight_layout()
plt.show()

## MATLAB to Python / NumPy Quick Reference

| Operation | MATLAB | Python (NumPy / SciPy) |
|-----------|--------|------------------------|
| Time vector | `t = 0:1/Fs:T-1/Fs;` | `t = np.arange(0, T, 1/Fs)` |
| Cosine | `cos(2*pi*f0*t)` | `np.cos(2*np.pi*f0*t)` |
| Complex exponential | `exp(1j*2*pi*f0*t)` | `np.exp(1j*2*np.pi*f0*t)` |
| Real / imaginary part | `real(x)` / `imag(x)` | `x.real` / `x.imag` |
| Envelope | `abs(x)` | `np.abs(x)` |
| White Gaussian noise | `randn(1, N)` | `np.random.randn(N)` |
| Signal power | `mean(abs(x).^2)` | `np.mean(np.abs(x)**2)` |
| FFT | `fft(x)` | `np.fft.fft(x)` |
| Centred FFT | `fftshift(fft(x))` | `np.fft.fftshift(np.fft.fft(x))` |
| Frequency axis (Hz) | `(-N/2:N/2-1)*Fs/N` | `np.fft.fftshift(np.fft.fftfreq(N, 1/Fs))` |
| dB magnitude | `20*log10(abs(X))` | `20*np.log10(np.abs(X))` |
| Linear chirp | `chirp(t, f0, T, f1)` | `scipy.signal.chirp(t, f0, f1, t1)` |
| Spectrogram | `spectrogram(x, win, nov, nfft, Fs)` | `scipy.signal.spectrogram(x, fs=Fs, nperseg=L, noverlap=H)` |